<a href="https://colab.research.google.com/github/ale-nunes/LLM/blob/main/Lab_2_Agentic_RAG_MultiAgent_BaseVendas_COMENTADO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB 2 — versão comentada para estudo

Este notebook é uma cópia didática do laboratório enviado, com explicações **célula por célula**.

Em cada etapa você verá:
1. O que a célula faz
2. Por que ela existe
3. Comentários dentro do código
4. Qual conceito de LLM/Agent ela representa

> Dica: leia a explicação, tente prever o resultado e só depois execute.

### O que esta célula faz
Instala todas as bibliotecas do laboratório no ambiente do Google Colab.

**Por que elas existem?**
- `langgraph`: constrói o workflow de agentes.
- `langchain` / `langchain-core`: abstrações de LLM, mensagens e tools.
- `langchain-google-genai`: conecta o LangChain ao Gemini.
- `pandas` / `numpy`: tratamento dos dados.
- `rank_bm25`: busca lexical.
- `sentence-transformers`: embeddings e reranking.
- `faiss-cpu`: banco/índice vetorial local.

**Ideia para memorizar:** esta célula prepara a “caixa de ferramentas” do projeto.

In [2]:
# Instala as dependências do laboratório no ambiente temporário do Google Colab.
# O sinal "!" executa um comando do sistema, não Python puro.
!pip -q install langgraph langchain langchain-core langchain-google-genai pandas numpy rank_bm25 sentence-transformers faiss-cpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.1 which is incompatible.


### O que esta célula faz
Importa as classes e funções que serão utilizadas no restante do laboratório.

O ponto mais importante é separar mentalmente os grupos:
1. **Dados:** `pandas`, `numpy`, `sqlite3`.
2. **LangChain:** `tool`, `Document`, mensagens e Gemini.
3. **LangGraph:** `StateGraph`, `START`, `END`, `MemorySaver`, `interrupt`, `Command`.
4. **Busca:** BM25, embeddings, CrossEncoder e FAISS.

`TypedDict` define o formato do estado do grafo.

`Annotated[list[BaseMessage], add]` diz ao LangGraph que a lista de mensagens deve ser acumulada usando `add`.

**Ideia para memorizar:** imports = “peças” que os próximos blocos vão montar.

In [3]:
# -----------------------------
# BIBLIOTECAS GERAIS
# -----------------------------
import os          # Acesso a variáveis de ambiente e recursos do sistema.
import json        # Conversão entre Python e JSON.
import sqlite3     # Banco SQLite local.
import numpy as np # Operações numéricas e arrays.

import pandas as pd # Manipulação da BaseVendas.

# TypedDict descreve o formato esperado do estado do LangGraph.
# Annotated permite adicionar metadados ao tipo.
from typing import TypedDict, Annotated

# add será usado para acumular mensagens no estado.
from operator import add

# @tool transforma funções em ferramentas que agentes podem chamar.
from langchain_core.tools import tool

# Document representa um pedaço de informação que pode ser recuperado pelo RAG.
from langchain_core.documents import Document

# Tipos de mensagens usadas entre usuário, modelo e agentes.
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

# Integração do LangChain com modelos Gemini.
from langchain_google_genai import ChatGoogleGenerativeAI

# Peças principais do LangGraph.
from langgraph.graph import StateGraph, START, END

# Checkpoint/memória do estado do workflow.
from langgraph.checkpoint.memory import MemorySaver

# interrupt pausa o workflow para intervenção humana.
# Command permite retomar o workflow.
from langgraph.types import interrupt, Command

# BM25 faz busca lexical por palavras.
from rank_bm25 import BM25Okapi

# SentenceTransformer cria embeddings.
# CrossEncoder faz reranking.
from sentence_transformers import SentenceTransformer, CrossEncoder

# FAISS armazena e pesquisa vetores.
import faiss


### O que esta célula faz
Permite enviar a `BaseVendas.csv` para o Colab e carregá-la em um DataFrame.

- `files.upload()` abre o seletor de arquivos.
- `next(iter(uploaded))` pega o nome do arquivo enviado.
- `pd.read_csv()` transforma o CSV em tabela.
- `sep=";"` indica separador por ponto e vírgula.
- `encoding="utf-8-sig"` ajuda com arquivos que possuem BOM.
- `df.shape` mostra linhas e colunas.
- `df.head()` mostra as primeiras linhas.

**Ideia para memorizar:** CSV → DataFrame → `df`.

In [4]:
# Importa o mecanismo de upload do Google Colab.
from google.colab import files

# Abre uma janela para o usuário selecionar a BaseVendas.csv.
uploaded = files.upload()

# Pega o nome do primeiro arquivo enviado.
FILE_NAME = next(iter(uploaded))

# Lê o CSV.
# sep=";" porque a base utiliza ponto e vírgula.
# utf-8-sig ajuda com arquivos CSV exportados com BOM.
df = pd.read_csv(FILE_NAME, sep=";", encoding="utf-8-sig")

# Mostra quantidade de linhas e colunas.
print(df.shape)

# Mostra algumas linhas para conferência.
display(df.head())


Saving BaseVendas.csv to BaseVendas.csv
(9994, 19)


,Categoria,Cidade,Pais,Customer Name,Data do Pedido,Código Nota Fiscal,Código Postal,Nome do Produto,Região,Segmento,Data de Envio,Tipo de Envio,Status do Envio,Estado,Subcategoria,Desconto %,Lucro,Quantidade,Valor da venda
0,Móveis,Henderson,United States,Claire Gute,8/11/2016,CA-2016-152156,42420,Bush Somerset Collection Bookcase,Sul,Pessoa Física,11/11/2016,Correios,Enviado no dia,Kentucky,Bookcases,"0,00",42.0,2,262.0
1,Móveis,Henderson,United States,Claire Gute,8/11/2016,CA-2016-152156,42420,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Sul,Pessoa Física,11/11/2016,Correios,Enviado no dia,Kentucky,Chairs,"0,00",220.0,3,732.0
2,Materiais de Escritório,Los Angeles,United States,Darrin Van Huff,12/6/2016,CA-2016-138688,90036,Self-Adhesive Address Labels for Typewriters b...,Oeste,Corporativo,16/6/2016,Correios,Envio tardio,California,Labels,"0,00",7.0,2,15.0
3,Móveis,Fort Lauderdale,United States,Sean O'Donnell,11/10/2015,US-2015-108966,33311,Bretford CR4500 Series Slim Rectangular Table,Sul,Pessoa Física,18/10/2015,Padrão,Envio tardio,Florida,Tables,"0,45",-383.0,5,958.0
4,Materiais de Escritório,Fort Lauderdale,United States,Sean O'Donnell,11/10/2015,US-2015-108966,33311,Eldon Fold 'N Roll Cart System,Sul,Pessoa Física,18/10/2015,Padrão,Envio tardio,Florida,Storage,"0,20",3.0,2,22.0


### O que esta célula faz
Cria uma função para tentar corrigir números no padrão brasileiro.

Exemplo:
`1.234,56` → `1234.56`

Ela:
1. copia o DataFrame;
2. olha as colunas do tipo texto;
3. testa as primeiras 100 células;
4. remove pontos usados como separador de milhar;
5. troca vírgula por ponto;
6. tenta converter para número;
7. só converte a coluna se mais de 80% dos exemplos parecerem numéricos.

**Por que isso é importante?**
SQL e cálculos precisam reconhecer uma coluna como número, não como texto.

**Ideia para memorizar:** normalização prepara os dados antes dos agentes trabalharem.

In [5]:
def normalize_dataframe(df):
    # Trabalhamos em uma cópia para não alterar inesperadamente o DataFrame original.
    df = df.copy()

    # Percorre todas as colunas.
    for col in df.columns:

        # Só tentamos converter colunas atualmente armazenadas como texto.
        if df[col].dtype == "object":

            # Pegamos uma amostra para descobrir se a coluna parece numérica.
            s = df[col].dropna().astype(str).head(100)

            # Exemplo brasileiro:
            # "1.234,56" -> "1234.56"
            x = pd.to_numeric(
                s.str.replace(".", "", regex=False)
                 .str.replace(",", ".", regex=False),
                errors="coerce"
            )

            # Se mais de 80% da amostra virou número,
            # consideramos a coluna provavelmente numérica.
            if x.notna().mean() > 0.8:

                # Faz a conversão da coluna inteira.
                df[col] = pd.to_numeric(
                    df[col].astype(str)
                      .str.replace(".", "", regex=False)
                      .str.replace(",", ".", regex=False),
                    errors="coerce"
                )

    # Retorna o DataFrame normalizado.
    return df

# Executa a função.
df = normalize_dataframe(df)


### O que esta célula faz
Cria uma pequena camada SQL usando SQLite.

`sqlite3.connect(":memory:")` cria um banco somente na memória do notebook.

`df.to_sql("vendas", ...)` transforma o DataFrame em uma tabela chamada `vendas`.

Depois aparece a primeira **Tool**:

`@tool` transforma a função `execute_sql()` em uma ferramenta que um agente pode chamar.

A função:
- aceita uma query;
- verifica se começa com `SELECT`;
- bloqueia operações perigosas como `DELETE`, `DROP`, `UPDATE`;
- executa a consulta;
- devolve JSON.

**Conceito central:** o LLM não precisa calcular diretamente. Ele pode gerar SQL e pedir para uma ferramenta executar o cálculo.

In [6]:
# Cria um SQLite apenas na memória RAM.
# Ao reiniciar o Colab, ele desaparece.
conn = sqlite3.connect(":memory:")

# Copia o DataFrame para uma tabela chamada "vendas".
df.to_sql("vendas", conn, index=False, if_exists="replace")


# @tool transforma a função em uma ferramenta disponível para agentes.
@tool
def execute_sql(query: str) -> str:
    # Esta docstring explica para o agente qual é a finalidade da ferramenta.
    """Executa SELECT somente leitura na tabela vendas."""

    # Remove espaços e coloca a query em minúsculas para validação.
    q = query.strip().lower()

    # Segurança básica:
    # só aceitamos SELECT e bloqueamos comandos de alteração.
    if (
        not q.startswith("select")
        or any(
            x in q
            for x in [
                "insert", "update", "delete",
                "drop", "alter", "create", "truncate"
            ]
        )
    ):
        return "ERRO: apenas SELECT é permitido."

    try:
        # Executa a query e transforma o resultado em JSON.
        result = pd.read_sql_query(query, conn)

        return result.to_json(
            orient="records",
            force_ascii=False
        )

    except Exception as e:
        # Em vez de quebrar todo o programa, devolvemos o erro.
        return "ERRO SQL: " + str(e)


### O que esta célula faz
Transforma cada linha da base em um `Document`.

Depois cria o mecanismo de busca semântica:
1. `SentenceTransformer` gera embeddings;
2. FAISS armazena os vetores;
3. BM25 é preparado para busca por palavras.

`embedding` é uma representação numérica do significado do texto.

**Por que transformar cada linha em Document?**
Porque o RAG precisa de unidades que possam ser recuperadas.

**Ideia para memorizar:**
linha → texto → embedding → vetor → índice.

In [7]:
# Lista que armazenará os Documents usados pelo RAG.
documents = []

# Cada linha da BaseVendas será transformada em um documento.
for idx, row in df.iterrows():

    # Converte a linha para texto:
    # "coluna1: valor1 | coluna2: valor2 | ..."
    text = " | ".join(
        f"{c}: {row[c]}"
        for c in df.columns
        if pd.notna(row[c])
    )

    # Document guarda o conteúdo e metadados.
    documents.append(
        Document(
            page_content=text,
            metadata={"row_id": int(idx)}
        )
    )

# Modelo responsável por transformar texto em vetores.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Lista apenas com o texto dos documentos.
texts = [d.page_content for d in documents]

# Gera um vetor para cada documento.
embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Cria índice FAISS.
# embeddings.shape[1] é a quantidade de dimensões de cada vetor.
index = faiss.IndexFlatIP(embeddings.shape[1])

# Adiciona os embeddings ao índice.
index.add(
    np.asarray(embeddings, dtype="float32")
)

# Cria o índice BM25.
# Aqui cada documento vira uma lista de palavras.
bm25 = BM25Okapi(
    [x.lower().split() for x in texts]
)

print("docs", len(documents))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

docs 9994


### O que esta célula faz
Implementa **Hybrid Search + RRF + Reranking**.

`hybrid_search()` combina:
- busca vetorial;
- BM25;
- Reciprocal Rank Fusion (RRF).

Depois `CrossEncoder` faz o reranking.

Fluxo:
```text
Pergunta
   ↓
Vector Search ──┐
                ├→ RRF → candidatos
BM25 ───────────┘
                  ↓
              Reranker
                  ↓
              melhores docs
```

A `hybrid_rag_search` transforma todo esse processo em uma Tool.

**Ideia para memorizar:**
retrieval encontra candidatos; reranker decide quais são os melhores.

In [8]:
def hybrid_search(query, k=10):
    # -----------------------------
    # 1. BUSCA VETORIAL
    # -----------------------------

    # Converte a pergunta em embedding.
    q = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    # Busca os k vetores mais semelhantes no FAISS.
    _, ids = index.search(
        np.asarray(q, dtype="float32"),
        k
    )

    # Converte IDs para inteiros e remove IDs inválidos.
    vec = [
        int(i)
        for i in ids[0]
        if i >= 0
    ]

    # -----------------------------
    # 2. BM25
    # -----------------------------

    # Calcula relevância lexical.
    bm_scores = bm25.get_scores(
        query.lower().split()
    )

    # Pega os k documentos com maior pontuação BM25.
    bm = list(
        np.argsort(bm_scores)[::-1][:k]
    )

    # -----------------------------
    # 3. RRF
    # -----------------------------

    # Dicionário que acumula a relevância das duas buscas.
    scores = {}

    # Quanto melhor a posição no vetor, maior a contribuição.
    for r, i in enumerate(vec):
        scores[i] = scores.get(i, 0) + 1 / (60 + r)

    # Faz o mesmo para BM25.
    for r, i in enumerate(bm):
        scores[i] = scores.get(i, 0) + 1 / (60 + r)

    # Ordena os documentos pelo score combinado.
    ids = [
        i
        for i, _ in sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ][:k]

    # Retorna os Documents.
    return [documents[i] for i in ids]


# CrossEncoder compara diretamente:
# pergunta + documento
# e calcula um score de relevância.
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


def rerank(query, docs, top_k=5):

    # Cria pares [pergunta, documento].
    pairs = [
        [query, d.page_content]
        for d in docs
    ]

    # Calcula relevância de cada par.
    scores = reranker.predict(pairs)

    # Junta documento + score e ordena do maior para o menor.
    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Retorna somente os top_k documentos.
    return [
        d
        for d, _ in ranked[:top_k]
    ]


# Transforma todo o pipeline de recuperação em uma Tool.
@tool
def hybrid_rag_search(query: str) -> str:
    """Busca com BM25 + vetor + RRF + reranking."""

    # Primeiro recupera candidatos.
    candidates = hybrid_search(query, 10)

    # Depois reordena os candidatos.
    ranked = rerank(query, candidates, 5)

    # Converte para JSON para facilitar o consumo pelo agente.
    return json.dumps(
        [
            {
                "row_id": d.metadata["row_id"],
                "content": d.page_content
            }
            for d in ranked
        ],
        ensure_ascii=False
    )


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

### O que esta célula faz
Solicita a chave da API do Gemini sem deixá-la visível diretamente na célula.

`getpass()` mostra um campo de entrada oculto.

**Boa prática:** em projetos reais, prefira Secrets/variáveis de ambiente em vez de escrever a chave no código.

In [9]:
# getpass permite digitar a API Key sem deixá-la visível na tela.
from getpass import getpass

# Solicita a chave do Gemini.
GEMINI_API_KEY = getpass("Digite sua API Key: ")


Digite sua API Key: ··········


### O que esta célula faz
Cria um cliente direto da biblioteca `google.genai`.

Isso serve como teste/conexão independente do LangChain.

**Por que ter os dois?**
Para estudo, ajuda a entender a diferença:
- SDK do Google → chama o Gemini diretamente.
- LangChain → fornece uma interface que podemos usar dentro de agentes e workflows.

In [10]:
# SDK oficial do Google GenAI.
from google import genai

# Cria um cliente direto para testar a conexão com o Gemini.
client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini conectado!")


✅ Gemini conectado!


### O que esta célula faz
É um teste opcional de chamada direta ao Gemini.

Está comentado de propósito.

Se descomentar, você verifica se:
- a chave funciona;
- o modelo responde;
- a conexão com a API está funcionando.

**Ideia para memorizar:** primeiro teste a infraestrutura; depois construa o agente.

In [ ]:
# Teste opcional da API.
# Mantemos comentado para não gastar uma chamada toda vez que o notebook é executado.

# response = client.models.generate_content(
#     model="gemini-3.5-flash-lite",
#     contents="Explique o que é inteligência artificial em uma frase."
# )

# print(response.text)


### O que esta célula faz
Cria o objeto `llm`, que será o modelo utilizado pelos agentes.

Depois define `AgentState`, o estado compartilhado pelo LangGraph.

Cada campo representa uma informação que pode passar entre nodes:
- `question`: pergunta do usuário.
- `route`: rota escolhida.
- `sql_result`: evidências vindas do SQL.
- `rag_result`: evidências vindas do RAG.
- `draft_answer`: resposta preliminar.
- `critique`: avaliação do Critic.
- `approved`: aprovação.
- `messages`: histórico de mensagens.
- `memory_context`: contexto de memória.

**Conceito central:** no LangGraph, o estado é a “mochila” que viaja de node para node.

In [11]:
# Cria a interface LangChain para conversar com o Gemini.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)


# AgentState é o "contrato" do estado que circula pelo LangGraph.
class AgentState(TypedDict):

    # Pergunta original do usuário.
    question: str

    # Decisão do Router.
    route: str

    # Evidência produzida pelo SQL Agent.
    sql_result: str

    # Evidência produzida pelo RAG Agent.
    rag_result: str

    # Resposta preliminar.
    draft_answer: str

    # Avaliação do Critic.
    critique: str

    # Resultado da aprovação humana.
    approved: bool

    # Histórico de mensagens.
    # add faz as mensagens serem acumuladas.
    messages: Annotated[
        list[BaseMessage],
        add
    ]

    # Memórias recuperadas para a execução.
    memory_context: str


### O que esta célula faz
Cria o **Router Agent**.

Ele recebe a pergunta e decide:
- `SQL`: pergunta quantitativa;
- `RAG`: pergunta contextual/textual;
- `HYBRID`: precisa dos dois.

O `HumanMessage` envia instruções e pergunta ao modelo.

O código depois normaliza `r.content`, porque respostas de modelos podem aparecer em formatos diferentes.

Se o modelo devolver uma categoria inesperada, o código usa `HYBRID` como fallback.

**Ideia para memorizar:** Router não resolve a pergunta; ele decide **quem deve resolvê-la**.

In [12]:
# Prompt que define o trabalho do Router.
ROUTER_PROMPT = """
Classifique em SQL, RAG ou HYBRID.

SQL para:
- soma
- média
- contagem
- ranking
- faturamento
- lucro

RAG para:
- descrição
- contexto

HYBRID para:
- cálculo + contexto

Responda somente SQL, RAG ou HYBRID.
"""


def router_node(state):
    # Envia ao modelo as regras e a pergunta.
    r = llm.invoke([
        HumanMessage(content=ROUTER_PROMPT),
        HumanMessage(content=state["question"])
    ])

    # Algumas integrações podem devolver content como lista.
    # Este trecho normaliza para texto.
    if isinstance(r.content, list):
        content_str = " ".join(
            [
                part.get("text", "")
                if isinstance(part, dict)
                else str(part)
                for part in r.content
            ]
        )
    else:
        content_str = str(r.content)

    # Normaliza o resultado.
    route = content_str.strip().upper()

    # Fallback: se o modelo responder algo inesperado,
    # usamos HYBRID.
    if route not in ["SQL", "RAG", "HYBRID"]:
        route = "HYBRID"

    # O node devolve apenas os campos do estado que foram alterados.
    return {
        "route": route,
        "messages": [
            AIMessage(
                content=f"Router: {route}"
            )
        ]
    }


### O que esta célula faz
Cria três especialistas.

**SQL Agent**
- recebe a pergunta;
- conhece tabela e colunas;
- pede ao LLM uma consulta SQL;
- remove blocos Markdown;
- executa a query usando a Tool.

**RAG Agent**
- chama `hybrid_rag_search`.

**Hybrid Agent**
- chama SQL e RAG;
- devolve as duas evidências.

**Conceito Multi-Agent:** cada agente tem uma responsabilidade menor e mais clara.

In [13]:
def sql_agent_node(state):
    # Monta o prompt com a tabela, colunas e pergunta.
    p = f"""
Você é SQL Agent.
Tabela: vendas.
Colunas: {list(df.columns)}.
Pergunta: {state["question"]}.

Gere somente SELECT SQLite.
Não invente colunas.
"""

    # Pede ao LLM para gerar a SQL.
    r = llm.invoke(p)

    # Normaliza o conteúdo retornado.
    if isinstance(r.content, list):
        content_str = " ".join(
            [
                part.get("text", "")
                if isinstance(part, dict)
                else str(part)
                for part in r.content
            ]
        )
    else:
        content_str = str(r.content)

    # Remove ```sql ... ``` caso o modelo use Markdown.
    sql = (
        content_str
        .replace("```sql", "")
        .replace("```", "")
        .strip()
    )

    # Executa a SQL através da Tool.
    result = execute_sql.invoke(sql)

    return {
        "sql_result": result,
        "messages": [
            AIMessage(content=sql)
        ]
    }


def rag_agent_node(state):
    # Chama a Tool de Hybrid Search.
    return {
        "rag_result": hybrid_rag_search.invoke(
            state["question"]
        ),
        "messages": [
            AIMessage(content="RAG executado")
        ]
    }


def hybrid_agent_node(state):
    # Executa o especialista SQL.
    a = sql_agent_node(state)

    # Executa o especialista RAG.
    b = rag_agent_node(state)

    # Junta as duas evidências.
    return {
        "sql_result": a["sql_result"],
        "rag_result": b["rag_result"],
        "messages": [
            AIMessage(content="Hybrid executado")
        ]
    }


### O que esta célula faz
Cria dois agentes de raciocínio/controle.

**Analyst**
Recebe os resultados dos especialistas e produz uma resposta baseada nas evidências.

**Critic**
Recebe pergunta, resposta e evidências e verifica:
- se a resposta responde;
- se os números têm suporte;
- se existe invenção;
- se há contradição.

O Critic retorna `APPROVED` ou `REVISE`.

**Ideia para memorizar:**
Especialistas produzem evidências → Analyst escreve → Critic verifica.

In [14]:
def analyst_node(state):
    # O Analyst não consulta a base diretamente.
    # Ele recebe evidências dos especialistas e escreve uma resposta.
    p = f"""
Responda somente com base nas evidências.

Pergunta:
{state["question"]}

SQL:
{state.get("sql_result", "")}

RAG:
{state.get("rag_result", "")}

Não invente números.
Se faltar evidência, diga isso.
"""

    r = llm.invoke(p)

    # Normaliza o conteúdo.
    if isinstance(r.content, list):
        a = " ".join(
            [
                part.get("text", "")
                if isinstance(part, dict)
                else str(part)
                for part in r.content
            ]
        )
    else:
        a = str(r.content)

    return {
        "draft_answer": a,
        "messages": [
            AIMessage(content=a)
        ]
    }


def critic_node(state):
    # O Critic recebe a resposta e as evidências.
    p = f"""
Verifique a resposta.

Pergunta:
{state["question"]}

Resposta:
{state["draft_answer"]}

SQL:
{state.get("sql_result", "")}

RAG:
{state.get("rag_result", "")}

Comece com APPROVED ou REVISE e explique.
"""

    r = llm.invoke(p)

    # Normaliza o conteúdo.
    if isinstance(r.content, list):
        c = " ".join(
            [
                part.get("text", "")
                if isinstance(part, dict)
                else str(part)
                for part in r.content
            ]
        )
    else:
        c = str(r.content)

    # Consideramos aprovado quando a resposta começa com APPROVED.
    approved = c.strip().upper().startswith("APPROVED")

    return {
        "critique": c,
        "approved": approved,
        "messages": [
            AIMessage(content=c)
        ]
    }


### O que esta célula faz
Apresenta três conceitos.

**Corrective RAG**
Reformula a pergunta e faz nova recuperação quando a primeira busca não é suficiente.

**Human-in-the-loop**
`interrupt()` pausa o workflow e entrega ao humano:
- pergunta;
- resposta preliminar;
- crítica.

Depois o humano pode continuar a execução.

**Final**
Coloca a resposta final nas mensagens.

**Ideia para memorizar:** o humano vira uma etapa real do grafo, não apenas um botão externo.

In [15]:
def corrective_rag_node(state):
    # Pede ao LLM uma versão melhor da pergunta.
    q = llm.invoke(
        "Melhore esta consulta para busca semântica "
        "e retorne somente a consulta: "
        + state["question"]
    ).content

    # Faz uma nova busca com a consulta corrigida.
    return {
        "rag_result": hybrid_rag_search.invoke(q)
    }


def human_review_node(state):
    # interrupt pausa o grafo.
    # O conteúdo abaixo será mostrado à etapa humana.
    decision = interrupt({
        "question": state["question"],
        "draft_answer": state["draft_answer"],
        "critique": state["critique"]
    })

    # Aceita tanto dict quanto boolean.
    approved = (
        decision.get("approved", False)
        if isinstance(decision, dict)
        else bool(decision)
    )

    return {
        "approved": approved
    }


def final_node(state):
    # Coloca a resposta final no histórico.
    return {
        "messages": [
            AIMessage(
                content=state["draft_answer"]
            )
        ]
    }


### O que esta célula faz
Cria uma memória persistente simples usando SQLite.

Diferencie:
- `MemorySaver` → checkpoint/estado do workflow;
- SQLite `memories` → informação que queremos guardar entre execuções.

`save_memory()` grava uma memória.

`load_memory()` recupera memórias de um usuário.

**Ideia para memorizar:** checkpoint guarda o estado do processo; long-term memory guarda informação persistente.

In [16]:
# Banco persistente simples para memórias.
# O arquivo fica no ambiente do Colab enquanto ele existir.
memory_conn = sqlite3.connect(
    "long_term_memory.db",
    check_same_thread=False
)

# Cria a tabela se ela ainda não existir.
memory_conn.execute(
    "CREATE TABLE IF NOT EXISTS memories "
    "(id INTEGER PRIMARY KEY AUTOINCREMENT, "
    "user_id TEXT, memory TEXT)"
)

memory_conn.commit()


def save_memory(user_id, memory):
    # Insere uma memória associada a um usuário.
    memory_conn.execute(
        "INSERT INTO memories(user_id,memory) VALUES (?,?)",
        (user_id, memory)
    )

    # Confirma a gravação.
    memory_conn.commit()


def load_memory(user_id):
    # Recupera todas as memórias daquele usuário.
    rows = memory_conn.execute(
        "SELECT memory FROM memories WHERE user_id=?",
        (user_id,)
    ).fetchall()

    # Junta várias memórias em um único texto.
    return "\n".join(
        r[0]
        for r in rows
    )


# Grava uma primeira memória de exemplo.
save_memory(
    "usuario-001",
    "Usuário está estudando LLM, RAG e LangGraph."
)


### O que esta célula faz
Cria um **subgraph**.

Esse subgrafo contém:
```text
Analyst → Critic
```

A vantagem é encapsular uma parte do workflow em um componente reutilizável.

Em projetos maiores, subgraphs ajudam a separar responsabilidades e organizar arquiteturas complexas.

In [17]:
# Cria um StateGraph separado para pesquisa/análise.
research = StateGraph(AgentState)

# Adiciona dois nodes.
research.add_node("analyst", analyst_node)
research.add_node("critic", critic_node)

# Define o fluxo.
research.add_edge(START, "analyst")
research.add_edge("analyst", "critic")
research.add_edge("critic", END)

# Compila o subgrafo.
research_subgraph = research.compile()

print("Subgraph criado")


Subgraph criado


### O que esta célula faz
Constrói o grafo principal.

Fluxo:
```text
START
 ↓
Router
 ├→ SQL
 ├→ RAG
 └→ HYBRID
       ↓
    Analyst
       ↓
    Critic
       ↓
 Human Review
       ↓
     Final
       ↓
      END
```

`add_conditional_edges()` faz o roteamento dinâmico.

`MemorySaver()` adiciona checkpoint ao grafo.

**Conceito central:** LangGraph é a orquestração explícita do fluxo entre agentes.

In [18]:
# Função que transforma a decisão do Router em nome de rota.
def route_after_router(state):
    return state["route"].lower()


# Cria o grafo principal.
b = StateGraph(AgentState)


# Registra todos os nodes.
for name, fn in [
    ("router", router_node),
    ("sql", sql_agent_node),
    ("rag", rag_agent_node),
    ("hybrid", hybrid_agent_node),
    ("analyst", analyst_node),
    ("critic", critic_node),
    ("human_review", human_review_node),
    ("final", final_node)
]:
    b.add_node(name, fn)


# Todo workflow começa no Router.
b.add_edge(START, "router")


# O Router escolhe dinamicamente um dos três especialistas.
b.add_conditional_edges(
    "router",
    route_after_router,
    {
        "sql": "sql",
        "rag": "rag",
        "hybrid": "hybrid"
    }
)


# Os três especialistas convergem para o Analyst.
for x in ["sql", "rag", "hybrid"]:
    b.add_edge(x, "analyst")


# Depois da análise vem a crítica.
b.add_edge("analyst", "critic")

# Depois da crítica entra a aprovação humana.
b.add_edge("critic", "human_review")

# Após o humano, produzimos a saída final.
b.add_edge("human_review", "final")

# Finaliza o workflow.
b.add_edge("final", END)


# MemorySaver permite checkpoint do estado do workflow.
graph = b.compile(
    checkpointer=MemorySaver()
)

# Mermaid ajuda a visualizar a arquitetura.
print(
    graph.get_graph().draw_mermaid()
)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	sql(sql)
	rag(rag)
	hybrid(hybrid)
	analyst(analyst)
	critic(critic)
	human_review(human_review)
	final(final)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	analyst --> critic;
	critic --> human_review;
	human_review --> final;
	hybrid --> analyst;
	rag --> analyst;
	router -.-> hybrid;
	router -.-> rag;
	router -.-> sql;
	sql --> analyst;
	final --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### O que esta célula faz
Executa o workflow.

`thread_id` identifica uma conversa/execução.

O estado inicial é preenchido e enviado para `graph.invoke()`.

Como existe `interrupt()`, o fluxo pode parar no Human-in-the-loop.

**Ideia para memorizar:** `thread_id` permite que o LangGraph saiba qual execução deve continuar.

In [19]:
# Configuração da conversa.
# O thread_id identifica a execução/conversa.
config = {
    "configurable": {
        "thread_id": "usuario-001"
    }
}

# Pergunta de teste.
question = "Qual foi o faturamento total?"

# Estado inicial.
state = {
    "question": question,
    "route": "",
    "sql_result": "",
    "rag_result": "",
    "draft_answer": "",
    "critique": "",
    "approved": False,
    "messages": [],
    "memory_context": load_memory("usuario-001")
}

# Executa o grafo.
# Como existe interrupt(), a execução pode pausar.
result = graph.invoke(
    state,
    config=config
)

print(result)


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

{'question': 'Qual foi o faturamento total?', 'route': 'SQL', 'sql_result': '[{"SUM(\\"Valor da venda\\")":1305159.1880000001}]', 'rag_result': '', 'draft_answer': 'Com base nas evidências fornecidas, o faturamento total foi de 1.305.159,19 (ou 1.305.159,188).', 'critique': '**APPROVED**\n\n**Explicação:** \nO resultado da consulta SQL para a soma do "Valor da venda" é `1305159.1880000001`. A resposta fornecida arredondou corretamente o valor para `1.305.159,19` e incluiu o valor exato com três casas decimais (`1.305.159,188`), mantendo total fidelidade aos dados obtidos pela consulta SQL.', 'approved': False, 'messages': [AIMessage(content='Router: SQL', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), AIMessage(content='SELECT SUM("Valor da venda") FROM vendas;', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), AIMessage(content='Com base nas evidências fornecidas, o faturamento total foi de 1.305.159,19 (ou 1.305.15

### O que esta célula faz
Mostra como retomar um grafo pausado.

`Command(resume={"approved": True})` envia a decisão humana para o node que estava interrompido.

Isso demonstra um dos principais conceitos de Human-in-the-loop no LangGraph:
```text
executa → pausa → humano decide → continua
```

In [ ]:
# Quando o grafo estiver pausado no interrupt,
# usamos Command(resume=...) para continuar.

# Exemplo de aprovação humana:
#
# result = graph.invoke(
#     Command(
#         resume={
#             "approved": True
#         }
#     ),
#     config=config
# )
#
# print(result["draft_answer"])


### O que esta célula faz
Executa várias perguntas somente contra o Router.

É um teste simples para observar se perguntas diferentes seguem rotas diferentes.

Isso é importante porque antes de testar o sistema inteiro devemos testar partes menores.

**Boa prática de engenharia:** teste componentes individualmente antes de testar o workflow completo.

In [20]:
# Perguntas de teste para avaliar somente o Router.
tests = [
    "Qual foi o faturamento total?",
    "Qual produto vendeu mais?",
    "Quais informações existem na base?",
    "Qual produto teve maior faturamento e quais registros estão relacionados?"
]

# Testamos cada pergunta sem executar o workflow inteiro.
for q in tests:

    # Estado mínimo necessário para o Router.
    s = {
        "question": q,
        "route": "",
        "sql_result": "",
        "rag_result": "",
        "draft_answer": "",
        "critique": "",
        "approved": False,
        "messages": [],
        "memory_context": ""
    }

    # Mostra a rota escolhida.
    print(
        q,
        "=>",
        router_node(s)["route"]
    )


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Qual foi o faturamento total? => SQL


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Qual produto vendeu mais? => SQL


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Quais informações existem na base? => HYBRID


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Qual produto teve maior faturamento e quais registros estão relacionados? => HYBRID


### O que esta célula faz
Lista exercícios para transformar o laboratório em um projeto mais completo.

A progressão sugerida é:
1. novas Tools;
2. Chart Agent;
3. Forecast Agent;
4. saída estruturada do Critic;
5. ciclo Corrective RAG;
6. PostgreSQL;
7. FastAPI + Streamlit.

Isso transforma o laboratório didático em uma arquitetura próxima de um projeto de portfólio.

In [ ]:
# DESAFIOS PARA CONTINUAR O LABORATÓRIO
#
# 1. Criar top_products() como Tool.
#    Objetivo: encapsular uma consulta recorrente em uma ferramenta.
#
# 2. Criar Chart Agent.
#    Objetivo: transformar resultados em gráficos.
#
# 3. Adicionar Forecast Agent.
#    Objetivo: criar um especialista para previsão.
#
# 4. Fazer o Critic devolver JSON:
#    {
#       "approved": false,
#       "confidence": 0.72,
#       "issues": ["..."]
#    }
#
# 5. Criar ciclo Corrective RAG:
#    retrieve -> rerank -> generate -> critic -> retrieve novamente
#
# 6. Trocar SQLite por PostgreSQL.
#
# 7. Expor o LangGraph via FastAPI
#    e criar uma interface Streamlit.


# 🧠 MAPA MENTAL FINAL

```text
                 PERGUNTA
                    |
                    v
                 ROUTER
              /     |      \
            SQL    RAG    HYBRID
             |      |       |
             |   BM25 +     |
             |   Vector     |
             |      |       |
             |   Reranker   |
              \     |      /
                    v
                 ANALYST
                    |
                    v
                  CRITIC
                    |
                    v
             HUMAN-IN-LOOP
                    |
                    v
                  FINAL

MEMÓRIA:
- MemorySaver = estado/checkpoint do workflow
- SQLite       = memória persistente

LANGGRAPH:
- State = memória do workflow
- Node = função/agente
- Edge = caminho
- Conditional Edge = decisão
- interrupt = pausa para humano
- Command(resume) = continua
```

## Pergunta-chave para cada conceito

**Tool:** o agente precisa chamar uma função externa?

**RAG:** preciso recuperar informação antes de gerar?

**Hybrid Search:** uma única estratégia de busca não é suficiente?

**Reranker:** entre os candidatos encontrados, quais são realmente os melhores?

**Router:** quem deve resolver essa pergunta?

**Multi-Agent:** vale separar especialistas?

**Critic:** como verifico a resposta antes de entregá-la?

**Memory:** o sistema precisa lembrar algo depois?

**Human-in-the-loop:** existe uma decisão que deve passar por uma pessoa?

**LangGraph:** como orquestro tudo isso em um fluxo controlado?
